Check the model names from Pytorch Computer Vision (Timm) library:

In [1]:
# Hyperparameters
MODEL = "convnext_nano.in12k_timm"
DEVICE_ID = 0
BATCH_SIZE = 512
IMAGE_SIZE = 224
EPOCHS = 100
LR = 1e-4
OPTIMIZER = "AdamW"
DATA_DIR = r'/home/c/choton/beemachine/datasets/Others/fish-vista'
LOG_DIR = f"./{MODEL}_logs/"
SEED = 42

In [2]:
# Pytorch vision package (Timm)
import timm

# List relevant models
timm_name = MODEL[:-5] # Removing _timm in the name
convnext_models = timm.list_models(f'{timm_name}*')
print(convnext_models)

pretrained_models = timm.list_models(f'{timm_name}*', pretrained=True)
print(pretrained_models)

[]
['convnext_nano.in12k', 'convnext_nano.in12k_ft_in1k']


In [3]:
# Pytorch packages
import torch
from torch import nn
# from torch.nn import functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Other packages
import os
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Seeding for consistant results
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [4]:
# Load the classification splits and check the shape
train_csv = os.path.join(DATA_DIR, r'classification_train.csv')
val_csv = os.path.join(DATA_DIR, r'classification_val.csv')
test_csv = os.path.join(DATA_DIR, r'classification_test.csv')
train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

classes = sorted(train_df['standardized_species'].unique())
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}
print(f'Shape of FishVista classification datasets,  train: {train_df.shape}, validation: {val_df.shape}, test): {test_df.shape}')
print(f'Columns of the test dataset:', list(test_df.columns))

Shape of FishVista classification datasets,  train: (39800, 17), validation: (6779, 17), test): (9781, 17)
Columns of the test dataset: ['filename', 'source_filename', 'original_format', 'arkid', 'family', 'source', 'owner', 'standardized_species', 'original_url', 'license', 'adipose_fin', 'pelvic_fin', 'barbel', 'multiple_dorsal_fin', 'file_name', 'verbatim_species', 'species']


/tmp/ipykernel_1055974/3037820290.py:5: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(train_csv)


In [5]:
class FishVistaDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(DATA_DIR, "Images", row["filename"])
        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[row["standardized_species"]]
        if self.transform:
            image = self.transform(image)
        return image, label

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    # transforms.RandomHorizontalFlip(),
    # transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = FishVistaDataset(train_df, class_to_idx, train_transform)
val_dataset = FishVistaDataset(val_df, class_to_idx, val_transform)
test_dataset = FishVistaDataset(test_df, class_to_idx, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: 1758 | Train: 39800 | Val: 6779 | Test: 9781


In [7]:
image, label = train_dataset[33]
label

768

In [8]:
# Get one batch from the train loader
iterr = iter(train_loader)
images, labels = next(iterr)

print("Batch image tensor shape:", images.shape)
print("Batch label tensor shape:", labels.shape)

Batch image tensor shape: torch.Size([512, 3, 224, 224])
Batch label tensor shape: torch.Size([512])


In [9]:
# Loading the model
device = torch.device(f"cuda:{DEVICE_ID}")
num_classes = len(classes)
model = timm.create_model(timm_name, pretrained=True, num_classes=num_classes)
model.to(device)

# Setting up the criterion, optimizer and scheduler
criterion = nn.CrossEntropyLoss()  #label_smoothing=0.1)
if OPTIMIZER == "SGD":
    optimizer = torch.optim.SGD(
        model.parameters(), 
        lr=LR, 
        momentum=0.9, 
        weight_decay=1e-4)
elif OPTIMIZER == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LR, 
        weight_decay=1e-4)
else:
    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=LR,       # typically around 0.256 / batch_size scaling, you can tune
        alpha=0.9,              # decay (smoothing constant)
        eps=0.001,              # numerical stability
        momentum=0.9,           # as in paper
        weight_decay=1e-5,      # L2 regularization
        centered=False          # same as paper’s setup
    )
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

In [10]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0, 0, 0

    pbar = tqdm(train_loader, desc="[Train]", position=0, leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="[Val]", position=0, leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [11]:
%%time

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...


 Epoch 1/100 | Train Loss: 3.4512 | Train Acc: 0.4331 | Val Loss: 3.2540 | Val Acc: 0.4923


 Epoch 2/100 | Train Loss: 1.6727 | Train Acc: 0.7043 | Val Loss: 2.3434 | Val Acc: 0.6076


 Epoch 3/100 | Train Loss: 0.9869 | Train Acc: 0.8215 | Val Loss: 1.8905 | Val Acc: 0.6617


 Epoch 4/100 | Train Loss: 0.5606 | Train Acc: 0.9057 | Val Loss: 1.5996 | Val Acc: 0.7115


...

 Epoch 96/100 | Train Loss: 0.0019 | Train Acc: 1.0000 | Val Loss: 1.2941 | Val Acc: 0.7716


 Epoch 97/100 | Train Loss: 0.0019 | Train Acc: 1.0000 | Val Loss: 1.2941 | Val Acc: 0.7718


 Epoch 98/100 | Train Loss: 0.0019 | Train Acc: 1.0000 | Val Loss: 1.2941 | Val Acc: 0.7715


 Epoch 99/100 | Train Loss: 0.0019 | Train Acc: 1.0000 | Val Loss: 1.2940 | Val Acc: 0.7718


 Epoch 100/100 | Train Loss: 0.0019 | Train Acc: 1.0000 | Val Loss: 1.2940 | Val Acc: 0.7718
Training complete!
CPU times: user 1h 2min 15s, sys: 18min 6s, total: 1h 20min 22s
Wall time: 15h 42min 52s


In [12]:
test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 1.1606 | Test Acc: 0.7880


: 